## Ensemble & Hard-Case Routing Strategy

This script implements multiple ensemble-based submission strategies built on
logit-level aggregation of fine-tuned Transformer models (RoBERTa and DeBERTa),
with increasing levels of sophistication and risk.

All methods operate **in logit space**, following standard practices in ensemble
learning for neural networks, and are designed to maximize macro-F1 performance
on the evaluation set.

---

### Models Used
The ensemble combines four base models:

- RoBERTa (max length 512)
- RoBERTa (max length 256)
- DeBERTa (max length 512)
- DeBERTa (max length 256)

Each model contributes its raw logits on the evaluation set.

---

### 1. Simple Average Ensemble
**File:** `submission_avg.csv`

- Computes the unweighted mean of logits across all models.
- Serves as a baseline and sanity check.
- Low risk, low variance.

---

### 2. Weighted Logit Ensemble
**File:** `submission_weighted.csv`

- Aggregates logits using architecture- and performance-aware weights.
- Stronger models (e.g. RoBERTa-512) receive higher weight.
- Balances model strength and diversity.

This represents the **main baseline ensemble**.

---

### 3. Architecture-Specific Ensembles
**Files:**
- `submission_roberta_only.csv`
- `submission_deberta_only.csv`

- Intra-architecture averaging (RoBERTa-only, DeBERTa-only).
- Useful as specialists and diagnostic baselines.
- RoBERTa-only often performs better on ambiguous or long-context samples.

---

### 4. Hard-Case Routing (Entropy-Based)
**File:** `submission_hardcase.csv`

- Computes prediction entropy from the weighted ensemble probabilities.
- Samples with entropy above a given percentile threshold are treated as *hard cases*.
- For these samples, predictions are delegated to a specialist model (RoBERTa).

This exploits model disagreement to selectively improve difficult predictions.

---

### 5. Weighted + Hard-Case + Architecture Routing
**File:** `submission_weighted_hardcase.csv`

This is the most advanced strategy.

- Default prediction: weighted logit ensemble.
- For hard cases (high entropy):
  - Compare RoBERTa vs DeBERTa confidence (max softmax probability).
  - Select the architecture with higher confidence for that sample.

This approach combines:
- ensemble robustness,
- uncertainty-aware routing,
- per-sample architecture selection.

It approximates a conditional ensemble strategy and represents the **final
submission candidate with the highest expected gain**.

---

### Notes
- No temperature scaling is applied, as logits on the full development set
  are not available for all models.
- The strategy prioritizes macro-F1 over calibration metrics.
- Multiple submissions are generated to explore different bias–variance trade-offs.

---

### Recommended Final Submission
`submission_weighted_hardcase.csv`

This variant consistently provides the best balance between stability and
selective specialization on hard samples.


In [ ]:
import numpy as np
import pandas as pd
from scipy.special import softmax
import os


EVAL_CSV = "../data/processed/evaluation_processed.csv"
OUT_DIR  = "../data/submission/ensamble_submission"

LOGITS = {
	"roberta_512": "roberta_processed/logits_roberta_processed_noseed_MAXLEN_512.npy",
	"roberta_256": "roberta_MAXLEN256/logits_roberta_processed_noseed_MAXLEN_256.npy",
	"deberta_512": "deberta_processed/logits_deberta_processed_noseed_MAXLEN_512.npy",
	"deberta_256": "deberta_MAXLEN256/logits_deberta_processed_noseed_MAXLEN_256.npy",
}

WEIGHTS = {
	"roberta_512": 0.30,
	"roberta_256": 0.25,
	"deberta_512": 0.30,
	"deberta_256": 0.15,
}

HARDCASE_PERCENTILE = 70   
N_CLASSES = 7

os.makedirs(OUT_DIR, exist_ok=True)


In [ ]:
#LOAD DATA
df_eval = pd.read_csv(EVAL_CSV)
ids = df_eval["Id"].values

logits = {}
for name, path in LOGITS.items():
	logits[name] = np.load(path)
	print(f"{name}: {logits[name].shape}")

N, C = next(iter(logits.values())).shape
assert C == N_CLASSES

In [ ]:
#UTILS
def save_submission(preds, name):
	pd.DataFrame({
		"Id": ids,
		"Predicted": preds.astype(int)
	}).to_csv(f"{OUT_DIR}/{name}.csv", index=False)

def entropy(p):
	return -np.sum(p * np.log(p + 1e-12), axis=1)


In [ ]:
#SIMPLE AVERAGE
avg_logits = np.mean(list(logits.values()), axis=0)
avg_preds  = avg_logits.argmax(axis=1)

save_submission(avg_preds, "submission_avg")


In [ ]:
#WEIGHTED AVERAGE
weighted_logits = np.zeros_like(avg_logits)
for name, w in WEIGHTS.items():
	weighted_logits += w * logits[name]

weighted_preds = weighted_logits.argmax(axis=1)
save_submission(weighted_preds, "submission_weighted")


In [ ]:
#Roberta Only
rob_logits = (
	0.5 * logits["roberta_512"] +
	0.5 * logits["roberta_256"]
)
rob_preds = rob_logits.argmax(axis=1)

save_submission(rob_preds, "submission_roberta_only")

In [ ]:
# DEberta only 

deb_logits = (
	0.5 * logits["deberta_512"] +
	0.5 * logits["deberta_256"]
)
deb_preds = deb_logits.argmax(axis=1)

save_submission(deb_preds, "submission_deberta_only")

In [ ]:
#HardCase Routing 
base_probs = softmax(weighted_logits, axis=1)
H = entropy(base_probs)
threshold = np.percentile(H, HARDCASE_PERCENTILE)

rob_probs = softmax(rob_logits, axis=1)

final_preds = np.where(
	H > threshold,
	rob_probs.argmax(axis=1),
	base_probs.argmax(axis=1)
)
save_submission(final_preds, "submission_hardcase")

In [ ]:
# WEIGHTED + HARD-CASE + ARCH ROUTING
deb_probs = softmax(deb_logits, axis=1)

rob_conf = rob_probs.max(axis=1)
deb_conf = deb_probs.max(axis=1)

hard_mask = H > threshold

final_preds = base_probs.argmax(axis=1)  # default = weighted ensemble

choose_rob = rob_conf > deb_conf

final_preds[hard_mask & choose_rob] = rob_probs.argmax(axis=1)[hard_mask & choose_rob]
final_preds[hard_mask & (~choose_rob)] = deb_probs.argmax(axis=1)[hard_mask & (~choose_rob)]

save_submission(final_preds, "submission_weighted_hardcase")